# L6 demo: a validation gate, and a windowed replay

Two ideas from this session, on the Intel Berkeley Lab readings we have used since
L3. First, **validation**: we write down what the data must satisfy as an executable
schema, run it as a gate, and watch it stop the pipeline when the data is wrong.
Then **streaming**: we measure how out of order the feed really is, and replay one
sensor as tumbling and sliding windows in event time.

The lesson of the first half is that the gate **fails loudly on purpose**. The raw
feed already breaks the schema, so several cells below are expected to report a
failure. Each is caught so the notebook still runs top to bottom.

> Data: [Intel Lab Data](https://db.csail.mit.edu/labdata/labdata.html), ~2.3M readings
> from 54 motes, carried over from L3/L4.

## 1. Load the readings

The same loader as L3/L4: a whitespace-separated export with no header, rows not in
time order, and short or malformed rows we skip. The data is fetched from a mirror on
first run and cached under `data/`.

In [ ]:
import io
import urllib.request
import warnings
import zipfile
from pathlib import Path

import pandas as pd
import pandera.pandas as pa

# pandas 2.3 + numpy 2.5 emit a DeprecationWarning on Timestamp + Timedelta
# arithmetic (the 'generic' timedelta unit). It does not affect results.
warnings.filterwarnings('ignore', message="The 'generic' unit for NumPy timedelta")

DATA = Path('data/data.txt')
URL = 'https://raw.githubusercontent.com/linsea423/Intel_Lab_Data/master/data.zip'
COLS = ['date', 'time', 'epoch', 'moteid',
        'temperature', 'humidity', 'light', 'voltage']


def load(path: Path = DATA) -> pd.DataFrame:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f'fetching {URL}')
        with urllib.request.urlopen(URL) as r:
            payload = r.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as z:
            path.write_bytes(z.read('data.txt'))
    df = pd.read_csv(path, sep=r'\s+', names=COLS, header=None,
                     engine='c', on_bad_lines='skip')
    df['ts'] = pd.to_datetime(df['date'] + ' ' + df['time'],
                              format='mixed', errors='coerce')
    df = df.dropna(subset=['ts', 'moteid'])
    df['moteid'] = df['moteid'].astype(int)
    return df[['moteid', 'ts', 'temperature', 'humidity', 'light', 'voltage']]


readings = load()
print(f'{len(readings):,} readings; {readings.moteid.nunique()} distinct mote ids '
      f'(54 motes were deployed); '
      f'{readings.ts.min().date()} to {readings.ts.max().date()}')
readings.head(3)

## 2. Write down what you expect

A [pandera](https://pandera.readthedocs.io/) schema is a set of executable checks. We
assert three kinds at once: the **types** of the columns, the **set** a mote id must
fall in (54 motes were deployed, so a valid id is 1 to 54), and the **physical** ranges
a reading must respect. A temperature outside 0 to 50 degrees C is outside the
instrument's range; a battery below 2.4 V produces readings L3 showed we cannot trust.

The schema is just data describing data. Nothing has been checked yet.

In [ ]:
schema = pa.DataFrameSchema(
    {
        'moteid':      pa.Column(int,   pa.Check.isin(range(1, 55))),
        'temperature': pa.Column(float, pa.Check.in_range(0, 50), nullable=True),
        'voltage':     pa.Column(float, pa.Check.ge(2.4),        nullable=True),
    }
)
schema

## 3. Run the gate on the raw feed

`schema.validate(df)` raises `SchemaError` on the **first** thing that is wrong and
stops. That is exactly what a gate at the top of a pipeline should do: refuse to pass
data it cannot vouch for. We catch the error only so the notebook keeps running; in a
pipeline you would let it halt the job.

In [ ]:
try:
    schema.validate(readings)
    print('the raw feed passed the gate')
except pa.errors.SchemaError as e:
    bad_ids = sorted(int(v) for v in e.failure_cases['failure_case'].unique())
    print('GATE STOPPED THE PIPELINE at the first failing check:')
    print(f"  column {e.schema.name!r} failed its 'valid mote id' check")
    print(f'  offending mote ids: {bad_ids}')

The gate stopped at the very first thing wrong, before it ever looked at a
temperature. Some readings are tagged with mote ids outside the deployed 1 to 54,
including plainly corrupt values like 65407. A **set check** is the simplest kind of
schema check, and here it catches data corruption for free. About 0.4% of rows are
affected, small, but exactly the kind of thing that silently breaks a `GROUP BY mote`.

## 4. Collect every failure on a sample

`lazy=True` does not stop at the first failure; it checks everything and raises a
`SchemaErrors` (plural) carrying a `failure_cases` table: which column, which check,
which value, and the row index. Collecting every failure over 2.3M rows would
materialize hundreds of thousands of cases, so we run the lazy check on a sample to
show the shape of the report.

In [ ]:
sample = readings.head(50_000)
try:
    schema.validate(sample, lazy=True)
    print('sample passed')
except pa.errors.SchemaErrors as e:
    fc = e.failure_cases
    print(f'{len(fc):,} failing cases in {len(sample):,} rows')
    print(fc['check'].value_counts().to_string())
    print()
    print(fc[['column', 'check', 'failure_case', 'index']].head(6).to_string(index=False))

## 5. Why the gate matters: the confident wrong number

Without the gate, the pipeline runs to completion and hands you an answer. The answer
is wrong, and nothing warns you. Compare the mean temperature over the **raw** feed
against the mean over only the **physically plausible** rows. The impossible values do
not announce themselves; they just move the number.

In [ ]:
raw_mean = readings['temperature'].mean()
plausible = readings[readings['temperature'].between(0, 50)]
clean_mean = plausible['temperature'].mean()
dropped = len(readings) - len(plausible)
print(f'raw mean temperature       {raw_mean:8.2f} C   (all {len(readings):,} rows)')
print(f'plausible mean temperature {clean_mean:8.2f} C   ({dropped:,} impossible rows removed, {dropped / len(readings):.1%})')

## 6. A controlled injection

On the real feed the failures are real, which makes it hard to point at one. So take a
slice we know is clean, inject two rows we know are bad, and confirm the gate catches
**exactly** those two: a temperature above the range and a voltage below the floor.

In [ ]:
clean = plausible[plausible['voltage'] >= 2.4].head(8).reset_index(drop=True)

bad = pd.DataFrame({
    'moteid':      [7, 12],
    'ts':          pd.to_datetime(['2004-03-01 00:00:00', '2004-03-01 00:00:31']),
    'temperature': [122.15, 24.0],   # row 0: impossible temperature
    'humidity':    [40.0, 40.0],
    'light':       [100.0, 100.0],
    'voltage':     [2.6, 1.91],      # row 1: battery below the floor
})
spiked = pd.concat([clean, bad], ignore_index=True)

try:
    schema.validate(spiked, lazy=True)
    print('no failures (unexpected)')
except pa.errors.SchemaErrors as e:
    print(e.failure_cases[['column', 'check', 'failure_case', 'index']].to_string(index=False))

The report names the two injected rows and nothing else: index 8 for the impossible
temperature, index 9 for the low voltage. That precision is what makes a schema worth
keeping. It does not just say the data is bad; it says which row, which rule, which
value.

## 7. What a failing check should do

A check that fails needs a policy. Three common ones, from strictest to most forgiving:

- **block**: raise and stop, so bad data never reaches a model or a report
- **warn**: log it and carry on, when you want to monitor but not act yet
- **quarantine**: route the bad rows aside and let the good rows flow

Quarantine is often the right call for a sensor feed: one dead mote should not stop the
floor. We split the raw feed into a clean stream and a quarantine table using the same
physical rules the schema encodes.

In [ ]:
# Match the schema's physical checks: a missing value is nullable, so it is not 'bad'.
temp_bad = readings['temperature'].notna() & ~readings['temperature'].between(0, 50)
volt_bad = readings['voltage'].notna() & (readings['voltage'] < 2.4)
bad = temp_bad | volt_bad
good = readings[~bad]
quarantine = readings[bad]
print(f'passed      {len(good):,} rows ({len(good) / len(readings):.1%})')
print(f'quarantined {len(quarantine):,} rows ({len(quarantine) / len(readings):.1%})')
print()
print(f'  low battery (<2.4 V)     {volt_bad.sum():,}')
print(f'  impossible temperature   {temp_bad.sum():,}')
overlap = (temp_bad & volt_bad).sum() / temp_bad.sum()
print(f'  of the impossible temps, {overlap:.0%} are also low-voltage: the same failure')

The two rules reject nearly the same rows: 96% of the impossible temperatures come
from motes whose batteries had drained, exactly as L3 found. A physical check does not
just flag a bad value; it points at the mechanism that produced it.

## 8. Streaming: how out of order is the feed?

Now the other half. A reading's **event time** is when it was measured; its
**processing time** is when we see it. Over a lossy sensor network the two diverge, so
readings arrive out of event-time order. We measure it directly: per mote, what
fraction of readings have a timestamp earlier than a reading already seen from that
mote. This is why windowed aggregates must group by event time, not arrival order.

In [ ]:
def out_of_order_fraction(df: pd.DataFrame) -> float:
    earlier_than_seen = (df.groupby('moteid')['ts']
                         .apply(lambda s: s < s.cummax())
                         .reset_index(level=0, drop=True))
    return float(earlier_than_seen.mean())

frac = out_of_order_fraction(readings)
print(f'{frac:.1%} of readings arrive out of event-time order')

## 9. Replay one sensor as windows

Windowing turns an endless feed into numbers you can act on. Take the busiest mote's
first eight hours and aggregate its temperature two ways over event time: a **tumbling**
window (fixed one-hour buckets, no overlap) and a **sliding** window (a one-hour
average recomputed every fifteen minutes). This is the notes' windowing figure,
reproduced from the data.

In [ ]:
clean_stream = readings[readings['temperature'].between(0, 50)]
mote = int(clean_stream['moteid'].value_counts().idxmax())
s = (clean_stream[clean_stream['moteid'] == mote]
     .sort_values('ts').set_index('ts')['temperature'])
s = s[s.index < s.index.min() + pd.Timedelta(hours=8)]

tumbling = s.resample('1h').mean()
sliding = s.resample('15min').mean().rolling(4, min_periods=1).mean()

print(f'mote {mote}: {len(s):,} readings over 8 hours\n')
print('tumbling 1 h windows (event time):')
print(tumbling.round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(s.index, s.values, '.', ms=3, color='0.8', label='raw readings (~31 s)')
ax.step(tumbling.index, tumbling.values, where='post', color='#c41230', lw=2.5,
        label='tumbling 1 h mean')
ax.plot(sliding.index, sliding.values, color='#1f5c99', lw=1.8,
        label='sliding 1 h mean, 15 min hop')
ax.set_ylabel('Temperature, C')
ax.set_title(f'One stream, two windows (mote {mote}, first 8 hours)')
ax.legend(frameon=False, fontsize=9)
plt.show()

---

## Takeaway

The gate turns a silent, confident wrong number into a loud, specific failure: which
row, which check, which value. The three policies (block, warn, quarantine) decide
what to do when it fails, and for a sensor feed quarantine usually wins. On the
streaming side, most readings arrive out of order, which is why a windowed aggregate
has to group by event time rather than by when the data showed up. Assignment **A3**
has you write a validation suite for a sensor feed and reason about a windowed
aggregate, so both halves start here.